# 🎯 03 — Customer Segmentation
## RFM Rule-Based + K-Means Clustering

Two segmentation approaches applied and compared:
- **Approach A**: Rule-based RFM scoring (business-interpretable)
- **Approach B**: K-Means clustering (data-driven)

---

In [ ]:
import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.insert(0, os.path.abspath('..'))
import config

pd.set_option('display.max_columns', 50)
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

## 1. Load Feature Matrix

In [ ]:
customer_features = pd.read_csv(os.path.join(config.DATA_PROCESSED, 'customer_features.csv'))
print(f"Loaded: {customer_features.shape[0]:,} customers × {customer_features.shape[1]} features")

Loaded: 96,096 customers × 34 features


## 2. Run Segmentation Pipeline

In [ ]:
from src import segmentation

segmented = segmentation.run(customer_features=customer_features)


PHASE 3: Customer Segmentation

  Approach A: Rule-Based RFM Segmentation
  RFM segment distribution:
rfm_segment
Needs Attention        40951
Potential Loyalists    15457
Loyal Customers        15062
At Risk                 9125
High Value At Risk      7011
Champions               6050
Dormant                 2440

  Approach B: K-Means Clustering
  Optimal k = 3 (silhouette = 0.579)

  Cluster profiles:
         recency_days  frequency  monetary  engagement_score  avg_order_value
cluster                                                                      
0                 0.0        0.0       0.0              28.2              0.0
1               242.0        1.0     108.1              28.5            108.1
2               246.5        1.0     961.0              28.3            961.0

  Comparing RFM vs K-Means segmentation...
  Adjusted Rand Index: 0.066

  ✓ Segmented customers → d:\Customer Profitability and Retention Analytics\data\processed\customer_segmented.csv


## 3. Segment Analysis

In [ ]:
# RFM Segment Profile Summary
seg_profile = segmented.groupby('rfm_segment').agg(
    count=('customer_id', 'count'),
    avg_recency=('recency_days', 'mean'),
    avg_frequency=('frequency', 'mean'),
    avg_monetary=('monetary', 'mean'),
    avg_engagement=('engagement_score', 'mean'),
).round(1)

seg_profile['pct'] = (seg_profile['count'] / seg_profile['count'].sum() * 100).round(1)
seg_profile = seg_profile.sort_values('avg_monetary', ascending=False)
print(seg_profile.to_string())

                     count  avg_recency  avg_frequency  avg_monetary  avg_engagement   pct
rfm_segment                                                                               
High Value At Risk    7011        388.8            1.0         255.4            27.6   7.3
Champions             6050         84.8            1.0         254.2            29.9   6.3
Loyal Customers      15062        149.0            1.0         169.3            28.7  15.7
At Risk               9125        388.4            1.0         145.9            27.7   9.5
Needs Attention      40951        262.0            1.0         110.3            28.2  42.6
Dormant               2440        471.5            1.0          34.5            27.6   2.5
Potential Loyalists  15457         46.4            0.6          26.8            29.3  16.1


In [ ]:
# Visual: Segment distribution treemap-style
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# RFM Segment sizes
seg_sizes = segmented['rfm_segment'].value_counts().sort_values()
colors = sns.color_palette('viridis', len(seg_sizes))
seg_sizes.plot(kind='barh', ax=axes[0], color=colors)
axes[0].set_title('RFM Segments — Customer Count', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Count')
for i, v in enumerate(seg_sizes.values):
    axes[0].text(v + 50, i, f'{v:,}', va='center', fontsize=10)

# RFM Score distribution
segmented['rfm_score'].hist(bins=13, ax=axes[1], color='#3498db', edgecolor='white', alpha=0.8)
axes[1].set_title('RFM Score Distribution', fontsize=14, fontweight='bold')
axes[1].set_xlabel('RFM Score (3-15)')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.show()

C:\Users\HP\AppData\Local\Temp\ipykernel_23024\3892413074.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [ ]:
# Segment-level revenue contribution
seg_revenue = segmented.groupby('rfm_segment')['monetary'].sum().sort_values(ascending=False)
seg_revenue_pct = (seg_revenue / seg_revenue.sum() * 100).round(1)

fig, ax = plt.subplots(figsize=(10, 6))
seg_revenue_pct.plot(kind='bar', ax=ax, color=sns.color_palette('magma', len(seg_revenue_pct)), edgecolor='white')
ax.set_title('Revenue Contribution by Segment (%)', fontsize=14, fontweight='bold')
ax.set_ylabel('% of Total Revenue')
ax.tick_params(axis='x', rotation=45)
for i, v in enumerate(seg_revenue_pct.values):
    ax.text(i, v + 0.5, f'{v}%', ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

C:\Users\HP\AppData\Local\Temp\ipykernel_23024\1348234859.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
**✅ Segmentation complete!** Proceed to `04_profitability.ipynb`.